In [ ]:
'''
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MINI PROJECT - DAY 6
Title: Department Summary Report
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Goal:
Use Linux text processing and Python dictionaries
together to produce a department-level summary
report from a flat CSV file.

---

STEP 1 - Linux Setup

mkdir day6_project
cd day6_project

Create employees.csv with the 8-row dataset
used in today's exercises.

---

STEP 2 - Linux Pre-processing

Before touching Python, use Linux to:

1. Verify total row count (excluding header).
2. Extract and list unique departments.
3. Sort the file by department and save to
   sorted_by_dept.csv using sort.

Commands: wc -l  cut  sort  uniq

---

STEP 3 - Python Task

Create summary.py inside day6_project.

The program must:

1. Read employees.csv using open() and readlines().
2. Skip the header line.
3. For each department, calculate:
   - Total number of employees
   - Total salary
   - Average salary
   - Highest paid employee name and salary
4. Store all results in a nested dictionary.
5. Print a clean formatted report.

Expected output format:

Department Summary Report
=========================

Department : IT
Employees  : 4
Total Sal  : Rs. 3,03,000
Avg Sal    : Rs. 75,750
Top Earner : Divya (Rs. 91,000)

Department : HR
Employees  : 2
Total Sal  : Rs. 1,00,000
Avg Sal    : Rs. 50,000
Top Earner : Sneha (Rs. 52,000)

(continue for Finance)

Concepts: open(), readlines(), split(), int(),
          nested dict, loop, f-string, round()

---

STEP 4 - SQL Verification

Create the same employees table in SQLite.
Run the combined query from SQL Exercise 4.

Verify that SQL output matches
your Python report exactly.

This confirms your Python logic is correct.

---

End Goal:
Grouping data manually in Python using dictionaries
is exactly what SQL GROUP BY does automatically.
Understanding both levels is what separates a good
data engineer from someone who only knows syntax.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DAILY TARGETS
60 minutes  -- Solve exercises
20 minutes  -- Review and discuss in group
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
'''

In [ ]:
# STEP 2 - Linux Pre-processing
tail -n +2 employees.csv | wc -l

awk 'NR>1' employees.csv | wc -l

cut -d',' -f3 employees.csv | sort | uniq

sort -t',' -k3,3 employees.csv > sort_dept.csv
cat sort_dept.csv

In [ ]:
# STEP 3 - Python Task Answer -

import pandas as pd
df = pd.read_csv("employees.csv")

# 2. Group by department and aggregate
summary = df.groupby("department").agg(
    count= ("name", "count"),
    total_salary=("salary", "sum"),
    avg_salary=("salary", "mean")
).reset_index()

# 3. Get highest paid employee per department
idx = df.groupby("department")["salary"].idxmax()
top_earners = df.loc[idx, ["department", "name", "salary"]]
top_earners = top_earners.rename(columns={
    "name": "highest_paid_name_emp",
    "salary": "highest_paid_salary_emp"
})

# 4. Merge both results
final_df = pd.merge(summary, top_earners, on="department")

# 5. Format output
print("Department Summary Report")
print("="*25)

for _, row in final_df.iterrows():
    print(f"Department : {row['department']}")
    print(f"Employees  : {row['count']}")
    print(f"Total sal  : Rs. {row['total_salary']}")
    print(f"Avg sal    : Rs. {round(row['avg_salary']):,}")
    print(f"Top Earner : {row['highest_paid_name_emp']} (Rs. {row['highest_paid_salary_emp']:,})")
    print()
    

In [ ]:
# STEP 4 - SQL Verification

SELECT
    DEPARTMENT,
    COUNT(*) AS EMP_COUNT,
    ROUND(AVG(SALARY), 0) AS AVERAGE_SAL,
    MAX(SALARY) AS HIGH_SALARY,
    MIN(SALARY) AS LOW_SALARY
FROM EMPLOYEES
GROUP BY DEPARTMENT
HAVING COUNT(*) > 1
ORDER BY AVERAGE_SAL DESC;

